# GEP-OnSSET GIS-Extraction Notebook for GEP-OnSSET

This is the GEP-OnSSET GIS extraction notebook that runs in bulk. 
First, browse and select each layer, then run the processing for all layers.

You may start by selecting this cell, then in the menu click **Run > Run All Cells**

### Useful hints and common error messages
* Make sure that all input layers are using EPSG:4326 as the coordinate system
* Make sure that the target "crs" is in a coordinate system using meters as the unit
* It is often useful to clip all the input layers to the country boundaries in order to reduce processing times
* Make sure that each dataset actually has some data within the country boundaries
* Some of the datasets require the user to choose values from a dropdown list below
* For hydro points and mini-grids, the vector layers need some specific column names to work
* In case a dataset still does not work, try opening it in QGIS and run the *Fix geometries* tool and save the new layer.
* If things do not work, it may be useful to go to the very top of this Jupyter Notebook and start again from cell 1

## Importing necessary packages (Mandatory)

Packages to be used are imported from the funcs.ipynb.

In [1]:
%run funcs.ipynb
import traceback
import time

# Step 1: Indicate the layers and parameters to be used

First, define the coordinate system (crs). 
Then select the correct layer each time. If there is a layer you do not have or wish to use, press **Cancel**

In [2]:
crs = 'EPSG:32628'       # already defined earlier in the notebook for this run
country_name = 'Guinea'  # set once per country,

In [3]:
messagebox.showinfo('OnSSET extraction', 'Output folder')
workspace = filedialog.askdirectory()

In [4]:
messagebox.showinfo('OnSSET', 'Select the admin boundaries')
admin = filedialog.askopenfilename(filetypes = (("vector",["*.shp", "*.gpkg", "*.geojson"]),("all files","*.*")))

In [5]:
messagebox.showinfo('OnSSET', 'Select the clusters')
clusters_path = filedialog.askopenfilename(filetypes = (("vector",["*.shp", "*.gpkg", "*.geojson"]),("all files","*.*")))
clusters = gpd.read_file(clusters_path)
clusters = clusters[~clusters.geometry.is_empty & clusters.geometry.notnull()].copy()
# Fix invalid geometries
clusters['geometry'] = clusters['geometry'].buffer(0)

# Create a unique id column if it doesn't already exist (required by zonal_stats_exact,
# processing_points, processing_lines, etc.)
clusters = clusters.reset_index(drop=True)
if 'id' not in clusters.columns:
    clusters['id'] = clusters.index + 1

In [6]:
## Raster layers
messagebox.showinfo('OnSSET', 'Select the Solar GHI layer')
ghi_layer = filedialog.askopenfilename(filetypes = (("rasters","*.tif"),("all files","*.*")))

In [7]:
messagebox.showinfo('OnSSET', 'Select the land Land Cover layer')
landcover_layer = filedialog.askopenfilename(filetypes = (("rasters","*.tif"),("all files","*.*")))

In [8]:
messagebox.showinfo('OnSSET', 'Select the Elevation (DEM) layer')
dem_layer = filedialog.askopenfilename(filetypes = (("rasters","*.tif"),("all files","*.*")))

In [9]:
messagebox.showinfo('OnSSET', 'Select the Travel Time layer')
travel_layer = filedialog.askopenfilename(filetypes = (("rasters","*.tif"),("all files","*.*")))

In [10]:
messagebox.showinfo('OnSSET', 'Select the Wind layer')
wind_layer = filedialog.askopenfilename(filetypes = (("rasters","*.tif"),("all files","*.*")))

In [11]:
messagebox.showinfo('OnSSET', 'Select the Night Lights layer')
ntl_layer = filedialog.askopenfilename(filetypes = (("rasters","*.tif"),("all files","*.*")))

In [12]:
messagebox.showinfo('OnSSET', 'Select the Population layer')
pop_layer = filedialog.askopenfilename(filetypes = (("rasters","*.tif"),("all files","*.*")))

In [13]:
messagebox.showinfo('OnSSET', 'Select the Custom Demand layer')
custDem_layer = filedialog.askopenfilename(filetypes = (("rasters","*.tif"),("all files","*.*")))

In [14]:
messagebox.showinfo('OnSSET', 'Select Health facilities layer')
health_path = filedialog.askopenfilename(
    filetypes=(("vector", ["*.shp", "*.gpkg", "*.geojson"]), ("all files", "*.*")))
preview_facility_types(health_path, 'DESCRIPTIF')

Found 9 unique values in "DESCRIPTIF":
 - Poste de sante
 - Officine de pharmacie
 - Centre de sante
 - Laboratoire (rattache FOSA)
 - Laboratoire (non rattache)
 - Centre de sante ameliore
 - Hopital
 - Point de vente (pharmacie)
 - Centre medical communal


['Poste de sante',
 'Officine de pharmacie',
 'Centre de sante',
 'Laboratoire (rattache FOSA)',
 'Laboratoire (non rattache)',
 'Centre de sante ameliore',
 'Hopital',
 'Point de vente (pharmacie)',
 'Centre medical communal']

In [15]:
# Cellule 1 : sélection du fichier
messagebox.showinfo('OnSSET', 'Select the Education facilities layer')
edu_path = filedialog.askopenfilename(
    filetypes=(("vector", ["*.shp", "*.gpkg", "*.geojson"]), ("all files", "*.*")))
preview_facility_types(edu_path, 'amenity')

Found 4 unique values in "amenity":
 - school
 - kindergarten
 - college
 - university


['school', 'kindergarten', 'college', 'university']

In [16]:
messagebox.showinfo('OnSSET', 'Select the Substations layer')
sub_layer = filedialog.askopenfilename(filetypes = (("vector",["*.shp", "*.gpkg", "*.geojson"]),("all files","*.*")))

In [17]:
messagebox.showinfo('OnSSET', 'Select the Existing HV layer')
HVexist_layer = filedialog.askopenfilename(filetypes = (("vector",["*.shp", "*.gpkg", "*.geojson"]),("all files","*.*")))

In [18]:
messagebox.showinfo('OnSSET', 'Select the Planned HV layer')
HVplan_layer = filedialog.askopenfilename(filetypes = (("vector",["*.shp", "*.gpkg", "*.geojson"]),("all files","*.*")))

In [19]:
messagebox.showinfo('OnSSET', 'Select the Existing MV layer')
MVexist_layer = filedialog.askopenfilename(filetypes = (("vector",["*.shp", "*.gpkg", "*.geojson"]),("all files","*.*")))

In [20]:
messagebox.showinfo('OnSSET', 'Select the Planned MV layer')
MVplan_layer = filedialog.askopenfilename(filetypes = (("vector",["*.shp", "*.gpkg", "*.geojson"]),("all files","*.*")))

In [21]:
messagebox.showinfo('OnSSET', 'Select the Roads layer')
road_layer = filedialog.askopenfilename(filetypes = (("vector",["*.shp", "*.gpkg", "*.geojson"]),("all files","*.*")))

In [22]:
messagebox.showinfo('OnSSET', 'Select the Distribution Transformer layer')
trx_layer = filedialog.askopenfilename(filetypes = (("vector",["*.shp", "*.gpkg", "*.geojson"]),("all files","*.*")))

In [23]:
messagebox.showinfo('OnSSET', 'Select the Hydro layer')
hydro_layer = filedialog.askopenfilename(filetypes = (("vector",["*.shp", "*.gpkg", "*.geojson"]),("all files","*.*")))

if hydro_layer != '':
    hydro=gpd.read_file(hydro_layer)
    
    messagebox.showinfo('OnSSET', 'Select the column which describes the hydropower POWER potential in each location')
    options = hydro.columns.tolist()
    hydropower = dropdown_popup(options)
    
    messagebox.showinfo('OnSSET', 'Select the UNIT of the power potential')
    options=['W', 'kW', 'MW']
    hydrounit = dropdown_popup(options)
    print(hydropower)
    print(hydrounit)

PowerMW
MW


In [24]:
messagebox.showinfo('OnSSET', 'Select the Mini Grid layer')
exist_MG_layer = filedialog.askopenfilename(filetypes = (("vector",["*.shp", "*.gpkg", "*.geojson"]),("all files","*.*")))

In [51]:
messagebox.showinfo('OnSSET', 'Select the Admin 1 layer')
adm1_layer = filedialog.askopenfilename(filetypes = (("vector",["*.shp", "*.gpkg", "*.geojson"]),("all files","*.*")))

if adm1_layer != '':
    admin_1_data = gpd.read_file(adm1_layer)
    messagebox.showinfo('OnSSET', 'Select the column which contains the Admin 1 level names')
    options = admin_1_data.columns.tolist()
    admin_col_name = dropdown_popup(options)
    clusters, admin_1_path = admin_1('Admin_1', admin_1_data, crs, workspace, clusters,
                                     admin_1_path=adm1_layer, admin_col_name=admin_col_name)

Processing finished: 2026-09-23 15:41:29


# Step 2: Process layers

## Import admin

In [26]:
admin = gpd.read_file(admin)

## Extract Global Horizontal Irradiation (GHI) from Raster layer

In [27]:
if ghi_layer != '':
    out, ghi_path = zonal_stats_exact('GHI', clusters, 'mean', ghi_layer)
    if type(out) == gpd.geodataframe.GeoDataFrame:
        clusters = out
else:
    print('GHI layer not selected')

Processing finished: 2026-09-23 14:20:22


## Extract Travel Time from Raster layer

In [28]:
if travel_layer != '':
    out, travel_path = zonal_stats_exact('TravelTime', clusters, 'mean', travel_layer)
    if type(out) == gpd.geodataframe.GeoDataFrame:
        clusters = out
else:
    print('Travel Time layer not selected')

Processing finished: 2026-09-23 14:21:17


## Extract Wind Velocity from Raster layer

In [29]:
if wind_layer != '':
    out, wind_path = zonal_stats_exact('WindVel', clusters, 'mean', wind_layer)
    if type(out) == gpd.geodataframe.GeoDataFrame:
        clusters = out
else:
    print('Wind Velocity layer not selected')

Processing finished: 2026-09-23 14:22:13


## Extract Land cover from Raster layer

In [30]:
if wind_layer != '':
    out, wind_path = zonal_stats_exact('LandCover', clusters, 'mean', landcover_layer)
    if type(out) == gpd.geodataframe.GeoDataFrame:
        clusters = out
else:
    print('land cover layer not selected')

Processing finished: 2026-09-23 14:23:08


## Extract Elevation from Raster layer

In [31]:
if dem_layer != '':
    clusters = processing_elevation_and_slope('Elevation', 'mean', clusters, workspace, crs, raster_path=dem_layer)
else:
    print('Elevation (DEM) layer not selected, skipping elevation and slope extraction')

Processing finished: 2026-09-23 14:24:07
Processing finished: 2026-09-23 14:25:08


## Extract Night Lights from Raster layer

In [32]:
if ntl_layer != '':
    out, ntl_path = zonal_stats_exact('NightLight', clusters, 'mean', ntl_layer)
    if type(out) == gpd.geodataframe.GeoDataFrame:
        clusters = out
else:
    print('NightLight layer not selected')

Processing finished: 2026-09-23 14:25:59


## Extract Population from Raster layer

In [33]:
if pop_layer != '':
    out, pop_path = zonal_stats_exact('Pop', clusters, 'sum', pop_layer)
    if type(out) == gpd.geodataframe.GeoDataFrame:
        clusters = out
else:
    print('Population layer not selected, skipping')

Processing finished: 2026-09-23 14:26:57


## Extract Custom Demand from Raster layer

In [34]:
if custDem_layer != '':
    out, cd_path = zonal_stats_exact('CustomDemand', clusters, 'mean', custDem_layer)
    if type(out) == gpd.geodataframe.GeoDataFrame:
        clusters = out
else:
    print('CustomDemand layer not selected')

Processing finished: 2026-09-23 14:27:51


## Preparing to run the vector data

In [35]:
clusters = preparing_for_vectors(workspace, clusters, crs)

Processing finished: 2026-09-23 14:27:56


## Health facilities

In [36]:
hc_mapping = {
    'Poste de sante':              'com_hc',
    'Centre de sante':             'primary_hc',
    'Centre de sante ameliore':    'primary_hc',
    'Centre medical communal':     'primary_hc',
    'Hopital':                     'referral_hc',
    # Non retenus comme générateurs de demande (exclus volontairement) :
    # 'Officine de pharmacie', 'Point de vente (pharmacie)',
    # 'Laboratoire (rattache FOSA)', 'Laboratoire (non rattache)'
}
clusters = process_health_facilities(clusters, crs, hc_mapping, health_path=health_path)

3552 facilities found contained within clusters
Finding nearest clusters for 173 unassigned facilities (within 2.0 km)
Assigned 172 facilities by proximity
2322/83010 clusters have a hc facility assigned
Processing finished: 2026-09-23 14:28:01


## Educational facilities

In [37]:
edu_mapping = {
    'kindergarten': 'sch_edu',   # regroupé avec 'school' : même palier de demande IRENA
    'school':       'sch_edu',
    'college':      'col_edu',
    'university':   'uni_edu',
}
clusters = process_educational_facilities(clusters, crs, edu_mapping, edu_path=edu_path)

1406 facilities found contained within clusters
Finding nearest clusters for 35 unassigned facilities (within 2.0 km)
Assigned 35 facilities by proximity
568/83010 clusters have a edu facility assigned
Processing finished: 2026-09-23 14:28:03


## Extract Distance from Substations (Vector point layer)

In [38]:
if sub_layer != '':
    out, substation_path = processing_points("Substation", admin, crs, workspace, clusters, False, sub_layer)
    if type(out) == gpd.geodataframe.GeoDataFrame:
        clusters = out
else:
    print('Substation layer not selected')

Processing finished: 2026-09-23 14:28:11


## Extract Distance from Existing high voltage lines (Vector line layer)

In [39]:
if HVexist_layer != '':
    out, existing_hv_path = processing_lines("Existing_HV", admin, crs, workspace, clusters, HVexist_layer)
    if type(out) == gpd.geodataframe.GeoDataFrame:
        clusters = out
else:
    print('Existing HV lines layer not selected')

Processing finished: 2026-09-23 14:28:25


## Extract Distance from Planned high voltage lines (Vector line layer)

In [40]:
if HVplan_layer != '':
    out, planned_hv_path = processing_lines("Planned_HV", admin, crs, workspace, clusters, HVplan_layer)
    if type(out) == gpd.geodataframe.GeoDataFrame:
        clusters = out
else:
    print('Planned HV lines layer not selected')

Processing finished: 2026-09-23 14:28:40


## Extract Distance from Existing medium voltage lines (Vector line layer) 

In [41]:
if MVexist_layer != '':
    out, existing_mv_path = processing_lines("Existing_MV", admin, crs, workspace, clusters, MVexist_layer)
    if type(out) == gpd.geodataframe.GeoDataFrame:
        clusters = out
else:
    print('Existing MV lines layer not selected')

Processing finished: 2026-09-23 14:29:10


## Extract Distance from Planned medium voltage lines (Vector line layer)

In [42]:
if MVplan_layer != '':
    out, planned_mv_path = processing_lines("Planned_MV", admin, crs, workspace, clusters, MVplan_layer)
    if type(out) == gpd.geodataframe.GeoDataFrame:
        clusters = out
else:
    print('Planned MV lines layer not selected')

Planned MV lines layer not selected


## Extract Distance from Roads (Vector line layer)

In [43]:
if road_layer != '':
    out, road_path = processing_lines("Road", admin, crs, workspace, clusters, road_layer)
    if type(out) == gpd.geodataframe.GeoDataFrame:
        clusters = out
else:
    print('Roads lines layer not selected')

Processing finished: 2026-09-23 14:33:35


## Extract Distance from Transformers (Vector point layer)

In [44]:
if trx_layer != '':
    out, trx_path = processing_points("Service Transformer", admin, crs, workspace, clusters, False, trx_layer)
    if type(out) == gpd.geodataframe.GeoDataFrame:
        clusters = out
else:
    print('Service transformer layer not selected')

Service transformer layer not selected


## Extract Distance from hydro points (Vector point layer)

In [45]:
if hydro_layer != '':
    out = hydro_bulk(admin, hydro, hydropower, hydrounit, crs, workspace, clusters)
    if type(out) == gpd.geodataframe.GeoDataFrame:
        clusters = out
else:
    print('Hydro points layer not selected')

Processing finished: 2026-09-23 14:33:45


## Extract Distance from Mini-Grid points (Vector point layer)

In [46]:
if exist_MG_layer != '':
    out, mg_path = processing_points("MG", admin, crs, workspace, clusters, False, exist_MG_layer)
    if type(out) == gpd.geodataframe.GeoDataFrame:
        clusters = out
else:
    print('Mini Grid layer not selected')

Mini Grid layer not selected


## Add admin 2 to the Cluster name

In [47]:
out, admin2_path = assign_admin2(clusters, workspace, crs, name_field='shapeName')
if type(out) == gpd.geodataframe.GeoDataFrame:
    clusters = out

300 settlements fell outside admin2 boundaries, assigning nearest region
NAME_2 assigned. Unique regions found: 34
Processing finished: 2026-09-23 14:34:08


## Conditioning & Export (Mandatory)

This is the final cell in the extraction. This cell has to be run.

In [52]:
#clusters = conditioning(clusters, workspace, 'Pop')
clusters = conditioning(clusters, workspace, 'Pop', crs=crs, country_name=country_name)
print('Workspace: ', workspace)

Processing finished: 2026-09-23 15:42:44
The extraction file is now ready for review & use in the workspace directory as 'OnSSET_InputFile.csv'!
Workspace:  C:/Users/Bachirou/Documents/gep_onsset_guinea/Gin_Output
